# Second baseline: UnLoc on Gibson(t)

Runs the authors' UnLoc code (ICLR 2026) with their released Gibson checkpoint on the
`gibson_t` test walks, at five walk lengths (100, 50, 35, 20, 15 frames), and puts our
numbers next to the paper's Table 1. Nothing is trained.

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**,
and approve the Google Drive pop-up.

- Reads `gibson_t.zip` and `desdf.zip` from `MyDrive/Thesis/` (the same zips as E1) and
  extracts only the 9 test scenes.
- Downloads the UnLoc checkpoint and the Depth Anything V2 Large backbone once and keeps
  them in `MyDrive/Thesis/E2/cache/`.
- Each walk length is one run. One run reports both table rows: *UnLoc* (with
  post-processing) and *UnLoc w/o post-processing*.
- Results go to `MyDrive/Thesis/E2/results/unloc_v1/`. If Colab disconnects, press
  **Run all** again: finished runs are skipped.

## 1. Settings

In [ ]:
import os, sys, json

UNLOC_REPO   = "https://github.com/matthias-wueest/UnLoc.git"
UNLOC_COMMIT = "3c7a301aa08049f6b1bb86f110e8e6e675e2300b"   # 21 Apr 2026
DA_COMMIT    = "a561b849ebae10a6f5ef49e26c83cbbcd36c71bf"   # Depth Anything V2 submodule, as pinned by UnLoc
UNLOC_CKPT_FOLDER = "https://drive.google.com/drive/folders/1Qb0IkKbqLyBpHbptTTX8WZI8NElTQZQc"
UNLOC_CKPT   = "unloc_gibson_vitl.ckpt"
DA_URL       = ("https://huggingface.co/depth-anything/Depth-Anything-V2-Metric-Hypersim-Large/"
                "resolve/main/depth_anything_v2_metric_hypersim_vitl.pth")
DA_CKPT      = "depth_anything_v2_metric_hypersim_vitl.pth"

T_LIST = [100, 50, 35, 20, 15]          # paper Table 1 walk lengths, most important first
RUN_TAG = "unloc_v1"

# Local test mode (outside Colab): a folder stands in for MyDrive, set by E2_TEST_DRIVE.
TEST_DRIVE = os.environ.get("E2_TEST_DRIVE")
if TEST_DRIVE:
    T_LIST = [int(t) for t in os.environ.get("E2_T_LIST", "100,20,15").split(",")]

CONTENT = os.environ.get("E2_CONTENT", "/content")
THESIS  = f"{TEST_DRIVE}/Thesis" if TEST_DRIVE else "/content/drive/MyDrive/Thesis"
CACHE   = f"{THESIS}/E2/cache"
RESULTS = f"{THESIS}/E2/results/{RUN_TAG}"
REPO    = f"{CONTENT}/UnLoc"
DATA    = f"{CONTENT}/data/Gibson_Floorplan_Localization_Dataset"
RUNS    = f"{CONTENT}/e2_runs"
print("walk lengths:", T_LIST)

walk lengths: [100, 50, 35, 20, 15]


## 2. Mount Google Drive

In [ ]:
if not TEST_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
for p in [f"{THESIS}/gibson_t.zip", f"{THESIS}/desdf.zip"]:
    assert os.path.exists(p), f"Not found: {p}. Put the zip in MyDrive/Thesis/ (same as E1)."
os.makedirs(f"{CACHE}/ckpt", exist_ok=True)
os.makedirs(f"{RESULTS}/logs", exist_ok=True)
print("Drive OK")

Mounted at /content/drive
Drive OK


## 3. GPU and packages

In [ ]:
import subprocess
if not TEST_DRIVE:
    # Lightning pinned as in E1: it loads checkpoints with full unpickling, which the
    # released checkpoints need on PyTorch 2.6+. gdown fetches the UnLoc checkpoint.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightning==2.5.6", "gdown"], check=True)
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime → Change runtime type → T4 GPU, then Run all again.")
    assert TEST_DRIVE, "A GPU is required on Colab."

GPU: Tesla T4


## 4. UnLoc code at a fixed version, plus our logging patch

In [ ]:
%%writefile {CONTENT}/unloc-e2-logging.patch
diff --git a/evaluate.py b/evaluate.py
index 41b8981..89934c5 100644
--- a/evaluate.py
+++ b/evaluate.py
@@ -54,6 +54,13 @@ from utils.geometry import (
 )
 from utils.postprocessing import postprocess_trajectory

+import csv  # [E2] per-trajectory and per-step CSVs
+
+# [E2] torch.cuda.synchronize() raises without a GPU; make it a no-op on CPU
+# so the script can be smoke-tested off-GPU. No effect when CUDA is present.
+if not torch.cuda.is_available():
+    torch.cuda.synchronize = lambda *a, **k: None
+
 # ---------------------------------------------------------------------------
 # Reproducibility
 # ---------------------------------------------------------------------------
@@ -217,6 +224,21 @@ def load_depth_network(args, device):
         net = UnLocDepthModule.load_from_checkpoint(
             checkpoint_path=args.checkpoint_path, strict=False,
         )
+
+    # [E2] strict=False hides missing weights. Check that every trained
+    # (non-backbone) weight of the model is in the checkpoint, and report the
+    # rest, so a silent partial load cannot pass as a result.
+    ckpt_keys = set(torch.load(args.checkpoint_path, map_location="cpu",
+                               weights_only=False)["state_dict"].keys())
+    model_keys = set(net.state_dict().keys())
+    backbone = lambda k: k.startswith("encoder.depth_feature.da.")
+    missing_head = sorted(k for k in model_keys - ckpt_keys if not backbone(k))
+    print(f"[E2] checkpoint keys: {len(ckpt_keys)}, model keys: {len(model_keys)}, "
+          f"backbone keys from checkpoint: {sum(backbone(k) for k in ckpt_keys)}, "
+          f"backbone keys from Depth Anything .pth: "
+          f"{sum(backbone(k) for k in model_keys - ckpt_keys)}, "
+          f"unexpected: {len(ckpt_keys - model_keys)}")
+    assert not missing_head, f"[E2] trained weights missing from checkpoint: {missing_head[:5]}"
     return net.to(device)


@@ -527,6 +549,12 @@ def main():
     # Motion model parameters for this dataset (paper settings, fixed).
     motion = MOTION_SIGMAS[args.dataset]

+    # [E2] Per-step timings and per-trajectory rows, kept in memory and
+    # written once after the loop (never inside the timed window).
+    step_rows = []
+    traj_rows = []
+    frames_evaluated = 0
+
     # ------------------------------------------------------------------
     # Loop over test trajectories
     # ------------------------------------------------------------------
@@ -663,6 +691,12 @@ def main():
             matching_time += t_match_end - t_match_start
             iteration_time += t_iter_end - t_iter_start
             n_iter += 1
+            # [E2] same timestamps, kept per step instead of only summed
+            step_rows.append((data_idx, t,
+                              (t_feat_end - t_feat_start) * 1e3,
+                              (t_match_end - t_match_start) * 1e3,
+                              (t_iter_end - t_iter_start) * 1e3))
+            frames_evaluated += 1

             # --- Per-frame metrics ---
             gt_depths_frame = np.array(data["gt_depth"])[t, :]
@@ -737,7 +771,8 @@ def main():
                 lr_xy=POSTPROC_LR_XY,
                 lr_theta=POSTPROC_LR_THETA,
             )
-            post_processing_time += time.time() - pp_start
+            pp_dur = time.time() - pp_start  # [E2] kept per trajectory
+            post_processing_time += pp_dur
             n_post_processing += 1

             refined_poses_map = refined_poses.cpu().numpy().copy()
@@ -762,6 +797,24 @@ def main():
             )
             refined_rmse_per_seq.append(refined_rmse)

+        # [E2] one row per trajectory (forward filter and post-processed)
+        i1 = SUCCESS_THRESHOLDS_M.index(1.0)
+        traj_rows.append(dict(
+            traj=data_idx, scene=scene, idx_in_scene=int(idx_within_scene),
+            frames=len(pred_poses_map),
+            fwd_success_1m=int(success_flags_per_seq[-1][i1]),
+            fwd_rmse_m=float(rmse_per_seq[-1]),
+            pp_success_1m=(int(refined_success_flags_per_seq[-1][i1])
+                           if args.postprocess else ""),
+            pp_rmse_m=(float(refined_rmse_per_seq[-1]) if args.postprocess else ""),
+            pp_ms=(pp_dur * 1e3 if args.postprocess else ""),
+            depth_mae_m=depth_mae_per_seq[-1],
+            depth_cos_sim=depth_cos_sim_per_seq[-1],
+            **{f"recall_{k}": float(v) for k, v in zip(
+                ["0.1m", "0.5m", "1m", "1m30deg", "2m", "5m", "10m"],
+                recalls_per_seq[-1])},
+        ))
+
     # ------------------------------------------------------------------
     # Convert accumulators to arrays
     # ------------------------------------------------------------------
@@ -822,6 +875,29 @@ def main():
         if n_post_processing > 0:
             print(f"    Avg post-processing time      : {post_processing_time / n_post_processing:.4f} s")

+    # ------------------------------------------------------------------
+    # [E2] Nothing dropped: every trajectory and every frame was evaluated
+    # ------------------------------------------------------------------
+    assert len(traj_rows) == N, f"[E2] trajectories evaluated {len(traj_rows)} != dataset {N}"
+    assert frames_evaluated == N * traj_l, \
+        f"[E2] frames evaluated {frames_evaluated} != {N} x {traj_l}"
+    if args.postprocess:
+        assert len(refined_rmse_per_seq) == N, "[E2] post-processing skipped a trajectory"
+    print(f"[E2] evaluated {len(traj_rows)}/{N} trajectories, "
+          f"{frames_evaluated}/{N * traj_l} frames")
+
+    # [E2] CSVs: one row per trajectory, one row per filter step
+    tag = f"T{traj_l}"
+    with open(os.path.join(output_dir, f"unloc_{tag}_traj.csv"), "w", newline="") as f:
+        w = csv.DictWriter(f, fieldnames=list(traj_rows[0].keys()) if traj_rows else ["traj"])
+        w.writeheader()
+        w.writerows(traj_rows)
+    with open(os.path.join(output_dir, f"unloc_{tag}_steps.csv"), "w", newline="") as f:
+        w = csv.writer(f)
+        w.writerow(["traj", "t", "feature_ms", "matching_ms", "iteration_ms"])
+        w.writerows(step_rows)
+    print(f"[E2] wrote unloc_{tag}_traj.csv and unloc_{tag}_steps.csv to {output_dir}")
+
     # ------------------------------------------------------------------
     # Save results
     # ------------------------------------------------------------------

Writing /content/unloc-e2-logging.patch


In [ ]:
def git(*a, cwd=None):
    return subprocess.run(["git", *a], cwd=cwd, check=True, capture_output=True, text=True).stdout.strip()

if not os.path.exists(f"{REPO}/.git"):
    git("clone", "-q", UNLOC_REPO, REPO)
git("checkout", "-q", "-f", UNLOC_COMMIT, cwd=REPO)
git("submodule", "update", "--init", "--recursive", "-q", cwd=REPO)
assert git("rev-parse", "HEAD", cwd=REPO) == UNLOC_COMMIT
assert git("rev-parse", "HEAD", cwd=f"{REPO}/depth_anything_v2") == DA_COMMIT, "Depth Anything submodule is not at the pinned commit"

# Apply the patch on a clean evaluate.py (safe to rerun).
git("checkout", "-q", "--", "evaluate.py", cwd=REPO)
git("apply", f"{CONTENT}/unloc-e2-logging.patch", cwd=REPO)
n = open(f"{REPO}/evaluate.py").read().count("[E2]")
print(f"UnLoc @ {UNLOC_COMMIT[:7]}, Depth Anything @ {DA_COMMIT[:7]}, patch applied ({n} [E2] tags)")

UnLoc @ 3c7a301, Depth Anything @ a561b84, patch applied (16 [E2] tags)


## 5. Checkpoints (downloaded once, then cached in Drive)

If a download fails: open the UnLoc checkpoint folder (link in the README), download
`unloc_gibson_vitl.ckpt`, put it in `MyDrive/Thesis/E2/cache/ckpt/`, and Run all again.
The same works for `depth_anything_v2_metric_hypersim_vitl.pth`.

In [ ]:
import hashlib, shutil, time, urllib.request

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for blk in iter(lambda: fh.read(1 << 20), b""):
            h.update(blk)
    return h.hexdigest()

def fetch_da(dst):
    tmp = dst + ".part"
    for attempt in range(5):
        try:
            urllib.request.urlretrieve(DA_URL, tmp)
            os.replace(tmp, dst)
            return
        except Exception as e:
            print(f"  Depth Anything download failed ({e}); retry {attempt + 1}/5")
            time.sleep(10)
    raise RuntimeError(f"Could not download {DA_CKPT}. Put it in {CACHE}/ckpt/ by hand and rerun.")

def fetch_unloc(dst):
    import gdown
    tmpdir = f"{CONTENT}/unloc_ckpt_dl"
    os.makedirs(tmpdir, exist_ok=True)
    try:
        # Download only the Gibson checkpoint, not the whole folder.
        items = gdown.download_folder(UNLOC_CKPT_FOLDER, skip_download=True, quiet=True)
        item = next(i for i in items if os.path.basename(i.path) == UNLOC_CKPT)
        gdown.download(id=item.id, output=f"{tmpdir}/{UNLOC_CKPT}", quiet=False, resume=True)
    except Exception as e:
        print(f"  single-file download failed ({e}); downloading the folder instead")
        gdown.download_folder(UNLOC_CKPT_FOLDER, output=tmpdir, quiet=False, resume=True)
    found = [os.path.join(r, f) for r, _, fs in os.walk(tmpdir) for f in fs if f == UNLOC_CKPT]
    if not found:
        raise RuntimeError(f"Could not download {UNLOC_CKPT}. Put it in {CACHE}/ckpt/ by hand and rerun.")
    shutil.copy(found[0], dst + ".part")
    os.replace(dst + ".part", dst)

for name, fetch, min_size in [(DA_CKPT, fetch_da, 1.0e9), (UNLOC_CKPT, fetch_unloc, 1.0e6)]:
    cached = f"{CACHE}/ckpt/{name}"
    if not (os.path.exists(cached) and os.path.getsize(cached) > min_size):
        print("downloading", name)
        fetch(cached)
    assert os.path.getsize(cached) > min_size, f"{cached} is too small; delete it and rerun"
    print(f"{name}: {os.path.getsize(cached) / 1e6:.0f} MB")

# Copy to the places the code expects (local disk is faster than Drive).
os.makedirs(f"{REPO}/depth_anything_v2/checkpoints", exist_ok=True)
os.makedirs(f"{REPO}/logs", exist_ok=True)
for src, dst in [(f"{CACHE}/ckpt/{DA_CKPT}", f"{REPO}/depth_anything_v2/checkpoints/{DA_CKPT}"),
                 (f"{CACHE}/ckpt/{UNLOC_CKPT}", f"{REPO}/logs/{UNLOC_CKPT}")]:
    if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src)):
        shutil.copy(src, dst)
CKPT_SHA = {n: sha256(f"{CACHE}/ckpt/{n}") for n in (UNLOC_CKPT, DA_CKPT)}
for k, v in CKPT_SHA.items():
    print(f"{k}: sha256 {v}")

downloading depth_anything_v2_metric_hypersim_vitl.pth
depth_anything_v2_metric_hypersim_vitl.pth: 1341 MB
downloading unloc_gibson_vitl.ckpt


Downloading...
From: https://drive.google.com/uc?id=18WdEYWeLfAdTsjy4nqlitSvu2xmFF7K1
To: /content/unloc_ckpt_dl/unloc_gibson_vitl.ckpt
100%|██████████| 15.0M/15.0M [00:00<00:00, 123MB/s] 


unloc_gibson_vitl.ckpt: 15 MB
unloc_gibson_vitl.ckpt: sha256 48883fbdbe2a3f1f0b4f513038ba3cc75f780656d4186fa55604906323d4aac4
depth_anything_v2_metric_hypersim_vitl.pth: sha256 6f82ff2bc543ac02ddff4aa31fa363676a8305dd3ccf04e80e2af115a044cb6d


## 6. Test scenes of `gibson_t`, and their floorplan lookups (`desdf`)

Reads the zips straight from Drive and extracts only the test scenes listed in
`split.yaml`. Every file is checked against the zip's own checksum while extracting.

In [ ]:
import zipfile, yaml

def open_zip(path):
    try:
        z = zipfile.ZipFile(path)
        return z, None
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"

def extract_members(zpath, members, strip, dest):
    """Extract members (dropping the `strip` prefix) with retries; skips files already done."""
    todo = list(members)
    for attempt in range(6):
        z, err = open_zip(zpath)
        if z is None:
            raise RuntimeError(f"{zpath}: {err}")
        try:
            infos = {i.filename: i for i in z.infolist()}
            while todo:
                m = todo[0]
                info = infos[m]
                out = os.path.join(dest, m[len(strip):])
                if not (os.path.exists(out) and os.path.getsize(out) == info.file_size):
                    os.makedirs(os.path.dirname(out), exist_ok=True)
                    with z.open(info) as src, open(out + ".part", "wb") as dst:   # CRC checked on read
                        shutil.copyfileobj(src, dst, 1 << 20)
                    os.replace(out + ".part", out)
                todo.pop(0)
            return
        except (OSError, zipfile.BadZipFile) as e:
            print(f"  read error ({e}); retry {attempt + 1}/6")
            time.sleep(15)
        finally:
            z.close()
    raise RuntimeError(f"Could not extract from {zpath} after retries")

ZIP_INFO = {}
for name in ["gibson_t", "desdf"]:
    p = f"{THESIS}/{name}.zip"
    z, err = open_zip(p)
    ZIP_INFO[name] = (f"{os.path.getsize(p)} bytes, {len(z.infolist())} files" if z else f"BAD ({err})")
    print(f"{name}.zip: {ZIP_INFO[name]}")
    assert z, f"{name}.zip is damaged or incomplete; upload it again."
    z.close()

READY = f"{DATA}/.e2_ready"
if not os.path.exists(READY):
    # gibson_t: find split.yaml whatever the folder layout inside the zip is
    z, _ = open_zip(f"{THESIS}/gibson_t.zip")
    names = [i.filename for i in z.infolist() if not i.is_dir()]
    splits = sorted((n for n in names if n.endswith("split.yaml")), key=len)
    assert splits, "gibson_t.zip has no split.yaml"
    prefix = splits[0][: -len("split.yaml")]
    test_scenes = yaml.safe_load(z.read(splits[0]))["test"]
    z.close()
    want = [splits[0]] + [n for n in names
                          if n.startswith(prefix) and n[len(prefix):].split("/")[0] in test_scenes]
    print(f"gibson_t: {len(test_scenes)} test scenes, extracting {len(want)} of {len(names)} files")
    extract_members(f"{THESIS}/gibson_t.zip", want, prefix, f"{DATA}/gibson_t")

    # desdf: <anything>/<scene>/desdf.npy
    z, _ = open_zip(f"{THESIS}/desdf.zip")
    dnames = [i.filename for i in z.infolist() if i.filename.endswith("/desdf.npy")]
    z.close()
    for s in test_scenes:
        hit = [n for n in dnames if n.split("/")[-2] == s]
        assert hit, f"desdf.zip has no desdf for test scene {s}"
        extract_members(f"{THESIS}/desdf.zip", [hit[0]], hit[0][: -len(f"{s}/desdf.npy")], f"{DATA}/desdf")
    open(READY, "w").write("ok\n")

# Check every test scene is complete (same counts as E1: 9 scenes, 4061 poses).
split = yaml.safe_load(open(f"{DATA}/gibson_t/split.yaml"))
SCENE_POSES = {}
for s in split["test"]:
    d = f"{DATA}/gibson_t/{s}"
    n_pose = sum(1 for _ in open(f"{d}/poses.txt"))
    n_depth = sum(1 for _ in open(f"{d}/depth40.txt"))
    n_rgb = len([f for f in os.listdir(f"{d}/rgb") if f.endswith(".png")])
    assert n_pose == n_depth == n_rgb and os.path.exists(f"{d}/map.png"), \
        f"{s}: poses {n_pose}, depths {n_depth}, images {n_rgb}"
    assert os.path.exists(f"{DATA}/desdf/{s}/desdf.npy"), f"{s}: no desdf"
    SCENE_POSES[s] = n_pose
EXPECTED_N = {T: sum(n // T for n in SCENE_POSES.values()) for T in T_LIST}
print(f"{len(SCENE_POSES)} test scenes, {sum(SCENE_POSES.values())} poses")
print("walks per length:", EXPECTED_N, "(paper: {100: 37, 50: 78, 35: 111, 20: 200, 15: 268})")

gibson_t.zip: 16429264687 bytes, 51609 files
desdf.zip: 29364839 bytes, 19 files
gibson_t: 9 test scenes, extracting 4089 of 51370 files
9 test scenes, 4061 poses
walks per length: {100: 37, 50: 78, 35: 111, 20: 200, 15: 268} (paper: {100: 37, 50: 78, 35: 111, 20: 200, 15: 268})


## 7. Record the setup (`provenance.txt`)

In [ ]:
import datetime, platform, numpy, scipy, cv2, torchvision, lightning
try:
    import xformers; xf = xformers.__version__
except Exception:
    xf = "not installed (Depth Anything uses plain PyTorch attention)"
prov = [
    f"session_start_utc: {datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')}",
    "platform: " + ("local test" if TEST_DRIVE else "Google Colab"),
    f"unloc_commit: {UNLOC_COMMIT}",
    f"depth_anything_v2_commit: {DA_COMMIT}",
    f"python: {platform.python_version()}",
    f"torch: {torch.__version__} (cuda {torch.version.cuda})",
    f"torchvision: {torchvision.__version__}  lightning: {lightning.__version__}  numpy: {numpy.__version__}"
    f"  scipy: {scipy.__version__}  opencv: {cv2.__version__}",
    f"xformers: {xf}",
    f"gpu: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (cpu)'}",
    *[f"ckpt {k}: sha256 {v}" for k, v in CKPT_SHA.items()],
    *[f"zip {k}: {v}" for k, v in ZIP_INFO.items()],
    f"gibson_t: {len(SCENE_POSES)} test scenes, {sum(SCENE_POSES.values())} poses, walks per length {EXPECTED_N}",
]
with open(f"{RESULTS}/provenance.txt", "a") as fh:
    fh.write("\n".join(prov) + "\n")
print("\n".join(prov))

session_start_utc: 2026-10-30T12:53:00+00:00
platform: Google Colab
unloc_commit: 3c7a301aa08049f6b1bb86f110e8e6e675e2300b
depth_anything_v2_commit: a561b849ebae10a6f5ef49e26c83cbbcd36c71bf
python: 3.13.16
torch: 2.11.0+cu130 (cuda 13.0)
torchvision: 0.26.0+cu130  lightning: 2.5.6  numpy: 2.1.3  scipy: 1.16.3  opencv: 5.0.0
xformers: not installed (Depth Anything uses plain PyTorch attention)
gpu: Tesla T4
ckpt unloc_gibson_vitl.ckpt: sha256 48883fbdbe2a3f1f0b4f513038ba3cc75f780656d4186fa55604906323d4aac4
ckpt depth_anything_v2_metric_hypersim_vitl.pth: sha256 6f82ff2bc543ac02ddff4aa31fa363676a8305dd3ccf04e80e2af115a044cb6d
zip gibson_t: 16429264687 bytes, 51609 files
zip desdf: 29364839 bytes, 19 files
gibson_t: 9 test scenes, 4061 poses, walks per length {100: 37, 50: 78, 35: 111, 20: 200, 15: 268}


## 8. Run UnLoc at each walk length

One run per walk length, about 4,000 frames each. Each finished run is copied to Drive
and skipped next time.

In [ ]:
def run_one(T):
    dst = f"{RESULTS}/T{T}"
    if os.path.exists(f"{dst}/DONE"):
        print(f"T={T}: already done, skipping")
        return
    out = f"{RUNS}/gibson_t/res0.1_orn36_T{T}_dt1"
    shutil.rmtree(out, ignore_errors=True)
    cmd = [sys.executable, "evaluate.py", "--dataset_path", DATA, "--dataset", "gibson_t",
           "--checkpoint_path", f"logs/{UNLOC_CKPT}", "--traj_len", str(T), "--output_dir", RUNS]
    log = f"{RESULTS}/logs/T{T}.log"
    print(f"T={T}: running (expect {EXPECTED_N[T]} walks) ...", flush=True)
    t0 = time.time()
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    last_progress = 0.0
    with open(log, "w") as fh:
        fh.write(" ".join(cmd) + "\n")
        p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, env=env)
        for line in p.stdout:          # text mode turns progress-bar updates into lines
            fh.write(line)
            fh.flush()
            if "%|" in line:            # progress bar: show at most one line a minute
                if "Evaluating" in line and time.time() - last_progress > 60:
                    print("   ", line.strip(), flush=True)
                    last_progress = time.time()
            elif line.strip():
                print("   ", line.rstrip(), flush=True)
        rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"T={T} failed (exit {rc}); see {log}")
    os.makedirs(dst, exist_ok=True)
    for f in [f"unloc_T{T}_traj.csv", f"unloc_T{T}_steps.csv", "results.npz"]:
        shutil.copy(f"{out}/{f}", f"{dst}/{f}")
    n = sum(1 for _ in open(f"{dst}/unloc_T{T}_traj.csv")) - 1
    assert n == EXPECTED_N[T], f"T={T}: {n} walks evaluated, expected {EXPECTED_N[T]}"
    open(f"{dst}/DONE", "w").write(f"{datetime.datetime.now().isoformat()} {time.time() - t0:.0f}s\n")
    print(f"T={T}: done in {(time.time() - t0) / 60:.1f} min, saved to {dst}")

for T in T_LIST:
    run_one(T)

T=100: running (expect 37 walks) ...
    xFormers not available
    xFormers not available
    ======= USING DEVICE: cuda =======
    Test set size: 37
    [E2] checkpoint keys: 22, model keys: 429, backbone keys from checkpoint: 0, backbone keys from Depth Anything .pth: 407, unexpected: 0
    Loading DeSDF ...
    Loading poses and maps ...
    Evaluating:  97%|█████████▋| 36/37 [38:00<01:03, 63.43s/it]
      Forward filter:
        Success @ 10.0 m : 1.0000
        Success @  5.0 m : 0.9730
        Success @  2.0 m : 0.9730
        Success @  1.0 m : 0.9730
        Success @  0.5 m : 0.9189
        Success @  0.3 m : 0.7568
        Success @  0.2 m : 0.5405
        RMSE (succeeded @ 1m)  : 0.1552
        RMSE (all)             : 0.2846
      Single-frame depth & recall:
        Mean depth MAE         : 0.2507
        Mean depth cos. sim.   : 0.9812
        Recall @ 0.1m        : 0.1970
        Recall @ 0.5m        : 0.6111
        Recall @ 1m          : 0.6470
        Recall @ 1m+30

## 9. Our numbers next to the paper's

In [ ]:
%%writefile {CONTENT}/compare_e2.py
"""Compare our UnLoc run on gibson_t with the UnLoc paper (arXiv 2509.11301v2).

Reads the per-trajectory and per-step CSVs written by the [E2] patch of
evaluate.py, one folder per sequence length, and writes a Markdown table with
OK / CHECK / STOP verdicts.

Usage: python compare_e2.py RESULTS_DIR [OUT.md] [provenance.txt]
RESULTS_DIR holds T100/, T50/, T35/, T20/, T15/ (each with unloc_T*_traj.csv,
unloc_T*_steps.csv and results.npz).
"""
import csv
import math
import os
import sys

import numpy as np

T_LIST = [100, 50, 35, 20, 15]

# Paper Table 1 (Gibson(t), SR@1m %). N = number of test trajectories.
PAPER_N = {100: 37, 50: 78, 35: 111, 20: 200, 15: 268}
PAPER_SR_PP = {100: 97.3, 50: 94.9, 35: 92.8, 20: 86.5, 15: 81.3}    # "UnLoc"
PAPER_SR_FWD = {100: 97.3, 50: 92.3, 35: 88.3, 20: 70.5, 15: 65.3}   # "w/o post-processing"
# RMSE at T=100 (m): (successful, all). Same for both rows in the paper.
PAPER_RMSE_PP = (0.16, 0.28)
PAPER_RMSE_FWD = (0.16, 0.28)
# Paper Table 6, Gibson(t), UnLoc single-frame (observation only) row.
PAPER_SINGLE = {"recall_0.1m": 19.7, "recall_0.5m": 61.1, "recall_1m": 64.7,
                "recall_1m30deg": 63.8, "recall_2m": 68.7, "recall_5m": 84.4,
                "recall_10m": 97.6}
PAPER_DEPTH = {"depth_mae_m": 0.25, "depth_cos_sim": 0.981}
# Paper Table 3 (Quadro RTX 6000, means per frame) and Sec. 4 post-processing.
PAPER_TIME_MS = {"feature_ms": 174, "matching_ms": 4, "iteration_ms": None}
PAPER_PP_MS = 960

# F3Loc on the same walks: our E1 run (T=100) and the F3Loc rows of UnLoc Table 1.
F3LOC_E1_T100 = {"F3Loc mono (E1, ours)": 89.2, "F3Loc fusion (E1, ours)": 94.6}
F3LOC_PAPER = {"F3Loc mono": {100: 89.2, 50: 70.5, 35: 55.9, 20: 34.0, 15: 28.4},
               "F3Loc fusion": {100: 94.6, 50: 94.6, 35: 69.4, 20: 46.0, 15: 41.8}}


def read_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def ci95(p, n):
    """Half-width of a normal-approximation 95% interval, in points."""
    if n == 0:
        return float("nan")
    p = p / 100
    return 196 * math.sqrt(max(p * (1 - p), 1 / (4 * n * n)) / n)


def verdict_rate(ours, paper, n):
    """Success rates move in steps of 100/n points (one trajectory)."""
    d = abs(ours - paper)
    if d <= 0.1 + 1e-9:
        return "OK"
    step = 100 / n
    if d <= max(2.0, step) + 0.05 + 1e-9:
        return "CHECK"
    return "STOP"


def verdict_value(ours, paper, ok, check):
    d = abs(ours - paper)
    if d <= ok + 1e-9:
        return "OK"
    if d <= check + 1e-9:
        return "CHECK"
    return "STOP"


def pct(rows, key):
    return 100 * sum(int(r[key]) for r in rows) / len(rows)


def quantiles(x):
    x = np.asarray(x, dtype=float)
    return [np.percentile(x, q) for q in (50, 99, 99.9)] + [x.max()]


def main():
    root = sys.argv[1]
    out = sys.argv[2] if len(sys.argv) > 2 else os.path.join(root, "E2_table.md")
    prov = sys.argv[3] if len(sys.argv) > 3 else os.path.join(root, "provenance.txt")

    runs = {}
    for T in T_LIST:
        d = os.path.join(root, f"T{T}")
        f = os.path.join(d, f"unloc_T{T}_traj.csv")
        if os.path.exists(f):
            runs[T] = dict(traj=read_csv(f),
                           steps=read_csv(os.path.join(d, f"unloc_T{T}_steps.csv")),
                           npz=os.path.join(d, "results.npz"))

    L = ["# E2 results: UnLoc released checkpoint vs. paper (Gibson(t))", ""]
    if os.path.exists(prov):
        L += ["```", open(prov).read().strip(), "```", ""]
    missing = [T for T in T_LIST if T not in runs]
    if missing:
        L += [f"**Not run yet:** T = {', '.join(map(str, missing))}.", ""]

    # --- consistency with the script's own saved arrays ---
    consistency = []
    for T, r in runs.items():
        if not os.path.exists(r["npz"]):
            consistency.append(f"T{T}: results.npz missing")
            continue
        z = np.load(r["npz"])
        i1 = list(z["success_thresholds_m"]).index(1.0)
        a = 100 * z["success_flags"][:, i1].mean()
        b = pct(r["traj"], "fwd_success_1m")
        c = 100 * z["refined_success_flags"][:, i1].mean() if z["refined_success_flags"].size else float("nan")
        e = pct(r["traj"], "pp_success_1m") if r["traj"][0]["pp_success_1m"] != "" else float("nan")
        same = abs(a - b) < 1e-9 and (math.isnan(c) and math.isnan(e) or abs(c - e) < 1e-9)
        consistency.append(f"T{T}: CSV {'matches' if same else 'DOES NOT match'} results.npz "
                           f"(forward {b:.1f} vs {a:.1f}, post-processed {e:.1f} vs {c:.1f})")

    # --- Table 1 ---
    L += ["## Sequential localization, SR@1m (paper Table 1)", "",
          "| T | N ours / paper | row | ours (%) | ±95% | paper (%) | delta | verdict |",
          "|---|---|---|---|---|---|---|---|"]
    verdicts = []
    for T in T_LIST:
        if T not in runs:
            continue
        rows = runs[T]["traj"]
        n = len(rows)
        nv = "OK" if n == PAPER_N[T] else "STOP"
        for label, key, paper in [("UnLoc w/o post-processing", "fwd_success_1m", PAPER_SR_FWD[T]),
                                  ("UnLoc", "pp_success_1m", PAPER_SR_PP[T])]:
            ours = pct(rows, key)
            v = verdict_rate(ours, paper, n)
            if nv == "STOP":
                v = "STOP"
            verdicts.append(v)
            L.append(f"| {T} | {n} / {PAPER_N[T]} | {label} | {ours:.1f} | {ci95(ours, n):.1f} | "
                     f"{paper:.1f} | {ours - paper:+.1f} | {v} |")
    L += ["", "Verdict: OK = within 0.1 pt; CHECK = off by at most one trajectory "
          "(100/N pt) or 2 pt; STOP = more, or a different number of trajectories.", ""]

    # --- RMSE at T=100 ---
    if 100 in runs:
        rows = runs[100]["traj"]
        L += ["## RMSE at T=100 (m, last 10 frames)", "",
              "| row | RMSE successful ours / paper | RMSE all ours / paper | verdict |",
              "|---|---|---|---|"]
        for label, sk, rk, paper in [("UnLoc w/o post-processing", "fwd_success_1m", "fwd_rmse_m", PAPER_RMSE_FWD),
                                     ("UnLoc", "pp_success_1m", "pp_rmse_m", PAPER_RMSE_PP)]:
            succ = [float(r[rk]) for r in rows if int(r[sk])]
            allr = [float(r[rk]) for r in rows]
            rs = float(np.mean(succ)) if succ else float("nan")
            ra = float(np.mean(allr))
            v = max([verdict_value(rs, paper[0], 0.01, 0.05), verdict_value(ra, paper[1], 0.01, 0.05)],
                    key=["OK", "CHECK", "STOP"].index)
            verdicts.append(v)
            L.append(f"| {label} | {rs:.2f} / {paper[0]:.2f} | {ra:.2f} / {paper[1]:.2f} | {v} |")
        L += ["", "RMSE is the script's own: per trajectory over the last 10 frames, then averaged. "
              "OK within 0.01 m, CHECK within 0.05 m.", ""]

        # --- Single frame (Table 6), from the T=100 run ---
        nfr = len(rows) * 100
        L += ["## Single-frame localization, observation only (paper Table 6, Gibson(t))", "",
              f"From the T=100 run ({nfr} frames). The paper does not state which trajectory length "
              "its Table 6 used, so these verdicts are a cross-check, not part of the main criterion.", "",
              "| metric | ours | paper | delta | verdict |", "|---|---|---|---|---|"]
        for k, paper in PAPER_SINGLE.items():
            ours = 100 * float(np.mean([float(r[k]) for r in rows]))
            L.append(f"| {k.replace('recall_', 'R@')} (%) | {ours:.1f} | {paper:.1f} | {ours - paper:+.1f} | "
                     f"{verdict_value(ours, paper, 0.1, 1.0)} |")
        for k, paper in PAPER_DEPTH.items():
            ours = float(np.mean([float(r[k]) for r in rows]))
            ok, chk = (0.005, 0.02) if k == "depth_mae_m" else (0.0005, 0.005)
            nd = 2 if k == "depth_mae_m" else 3
            L.append(f"| {k} | {ours:.{nd}f} | {paper:.{nd}f} | {ours - paper:+.{nd}f} | "
                     f"{verdict_value(ours, paper, ok, chk)} |")
        L.append("")

    # --- Same table, extended: F3Loc and UnLoc on the same walks ---
    L += ["## F3Loc and UnLoc on the same gibson_t walks (SR@1m %)", "",
          "| method | source | " + " | ".join(f"T={T}" for T in T_LIST) + " |",
          "|---|---|" + "---|" * len(T_LIST)]
    for name, v in F3LOC_E1_T100.items():
        L.append(f"| {name.split(' (')[0]} | ours (E1) | {v:.1f} | " + " | ".join("–" for _ in T_LIST[1:]) + " |")
    for name, d in F3LOC_PAPER.items():
        L.append(f"| {name} | UnLoc paper | " + " | ".join(f"{d[T]:.1f}" for T in T_LIST) + " |")
    for label, key in [("UnLoc w/o post-processing", "fwd_success_1m"), ("UnLoc", "pp_success_1m")]:
        cells = [f"{pct(runs[T]['traj'], key):.1f}" if T in runs else "–" for T in T_LIST]
        L.append(f"| {label} | ours (E2) | " + " | ".join(cells) + " |")
    L.append("")

    # --- Timing (context only) ---
    L += ["## Per-step timing (context only; ours on Colab, paper on a Quadro RTX 6000)", "",
          "Lab rule: distributions only. The paper column is its reported mean, for orientation.", "",
          "| T | stage | N steps | p50 (ms) | p99 (ms) | p99.9 (ms) | max (ms) | paper mean (ms) |",
          "|---|---|---|---|---|---|---|---|"]
    for T in T_LIST:
        if T not in runs:
            continue
        st = runs[T]["steps"]
        for k in ["feature_ms", "matching_ms", "iteration_ms"]:
            q = quantiles([float(s[k]) for s in st])
            p = PAPER_TIME_MS[k]
            L.append(f"| {T} | {k[:-3]} | {len(st)} | " + " | ".join(f"{v:.2f}" for v in q) +
                     f" | {p if p is not None else 'n/a'} |")
        pp = [float(r["pp_ms"]) for r in runs[T]["traj"] if r["pp_ms"] != ""]
        if pp:
            q = quantiles(pp)
            L.append(f"| {T} | post-processing (per walk) | {len(pp)} | " +
                     " | ".join(f"{v:.1f}" for v in q) + f" | {PAPER_PP_MS} |")
    L.append("")

    n_ok = verdicts.count("OK")
    n_chk = verdicts.count("CHECK")
    n_stop = verdicts.count("STOP")
    L += ["## Summary", "",
          f"Main criterion (Table 1 rates and T=100 RMSE): {n_ok} OK, {n_chk} CHECK, {n_stop} STOP.", ""]
    L += [f"- {c}" for c in consistency]
    L.append("")

    open(out, "w").write("\n".join(L))
    print("\n".join(L))


if __name__ == "__main__":
    main()

Writing /content/compare_e2.py


In [ ]:
r = subprocess.run([sys.executable, f"{CONTENT}/compare_e2.py", RESULTS,
                    f"{RESULTS}/E2_table.md", f"{RESULTS}/provenance.txt"],
                   capture_output=True, text=True)
assert r.returncode == 0, r.stderr
from IPython.display import Markdown, display
display(Markdown(open(f"{RESULTS}/E2_table.md").read()))
print(f"Saved: {RESULTS}/E2_table.md")

# E2 results: UnLoc released checkpoint vs. paper (Gibson(t))

```
session_start_utc: 2026-10-30T12:53:00+00:00
platform: Google Colab
unloc_commit: 3c7a301aa08049f6b1bb86f110e8e6e675e2300b
depth_anything_v2_commit: a561b849ebae10a6f5ef49e26c83cbbcd36c71bf
python: 3.13.16
torch: 2.11.0+cu130 (cuda 13.0)
torchvision: 0.26.0+cu130  lightning: 2.5.6  numpy: 2.1.3  scipy: 1.16.3  opencv: 5.0.0
xformers: not installed (Depth Anything uses plain PyTorch attention)
gpu: Tesla T4
ckpt unloc_gibson_vitl.ckpt: sha256 48883fbdbe2a3f1f0b4f513038ba3cc75f780656d4186fa55604906323d4aac4
ckpt depth_anything_v2_metric_hypersim_vitl.pth: sha256 6f82ff2bc543ac02ddff4aa31fa363676a8305dd3ccf04e80e2af115a044cb6d
zip gibson_t: 16429264687 bytes, 51609 files
zip desdf: 29364839 bytes, 19 files
gibson_t: 9 test scenes, 4061 poses, walks per length {100: 37, 50: 78, 35: 111, 20: 200, 15: 268}
```

## Sequential localization, SR@1m (paper Table 1)

| T | N ours / paper | row | ours (%) | ±95% | paper (%) | delta | verdict |
|---|---|---|---|---|---|---|---|
| 100 | 37 / 37 | UnLoc w/o post-processing | 97.3 | 5.2 | 97.3 | -0.0 | OK |
| 100 | 37 / 37 | UnLoc | 97.3 | 5.2 | 97.3 | -0.0 | OK |
| 50 | 78 / 78 | UnLoc w/o post-processing | 92.3 | 5.9 | 92.3 | +0.0 | OK |
| 50 | 78 / 78 | UnLoc | 94.9 | 4.9 | 94.9 | -0.0 | OK |
| 35 | 111 / 111 | UnLoc w/o post-processing | 88.3 | 6.0 | 88.3 | -0.0 | OK |
| 35 | 111 / 111 | UnLoc | 92.8 | 4.8 | 92.8 | -0.0 | OK |
| 20 | 200 / 200 | UnLoc w/o post-processing | 70.5 | 6.3 | 70.5 | +0.0 | OK |
| 20 | 200 / 200 | UnLoc | 86.5 | 4.7 | 86.5 | +0.0 | OK |
| 15 | 268 / 268 | UnLoc w/o post-processing | 65.3 | 5.7 | 65.3 | -0.0 | OK |
| 15 | 268 / 268 | UnLoc | 81.3 | 4.7 | 81.3 | +0.0 | OK |

Verdict: OK = within 0.1 pt; CHECK = off by at most one trajectory (100/N pt) or 2 pt; STOP = more, or a different number of trajectories.

## RMSE at T=100 (m, last 10 frames)

| row | RMSE successful ours / paper | RMSE all ours / paper | verdict |
|---|---|---|---|
| UnLoc w/o post-processing | 0.16 / 0.16 | 0.28 / 0.28 | OK |
| UnLoc | 0.20 / 0.16 | 0.33 / 0.28 | CHECK |

RMSE is the script's own: per trajectory over the last 10 frames, then averaged. OK within 0.01 m, CHECK within 0.05 m.

## Single-frame localization, observation only (paper Table 6, Gibson(t))

From the T=100 run (3700 frames). The paper does not state which trajectory length its Table 6 used, so these verdicts are a cross-check, not part of the main criterion.

| metric | ours | paper | delta | verdict |
|---|---|---|---|---|
| R@0.1m (%) | 19.7 | 19.7 | +0.0 | OK |
| R@0.5m (%) | 61.1 | 61.1 | +0.0 | OK |
| R@1m (%) | 64.7 | 64.7 | +0.0 | OK |
| R@1m30deg (%) | 63.8 | 63.8 | -0.0 | OK |
| R@2m (%) | 68.7 | 68.7 | +0.0 | OK |
| R@5m (%) | 84.4 | 84.4 | -0.0 | OK |
| R@10m (%) | 97.6 | 97.6 | +0.0 | OK |
| depth_mae_m | 0.25 | 0.25 | +0.00 | OK |
| depth_cos_sim | 0.981 | 0.981 | +0.000 | OK |

## F3Loc and UnLoc on the same gibson_t walks (SR@1m %)

| method | source | T=100 | T=50 | T=35 | T=20 | T=15 |
|---|---|---|---|---|---|---|
| F3Loc mono | ours (E1) | 89.2 | – | – | – | – |
| F3Loc fusion | ours (E1) | 94.6 | – | – | – | – |
| F3Loc mono | UnLoc paper | 89.2 | 70.5 | 55.9 | 34.0 | 28.4 |
| F3Loc fusion | UnLoc paper | 94.6 | 94.6 | 69.4 | 46.0 | 41.8 |
| UnLoc w/o post-processing | ours (E2) | 97.3 | 92.3 | 88.3 | 70.5 | 65.3 |
| UnLoc | ours (E2) | 97.3 | 94.9 | 92.8 | 86.5 | 81.3 |

## Per-step timing (context only; ours on Colab, paper on a Quadro RTX 6000)

Lab rule: distributions only. The paper column is its reported mean, for orientation.

| T | stage | N steps | p50 (ms) | p99 (ms) | p99.9 (ms) | max (ms) | paper mean (ms) |
|---|---|---|---|---|---|---|---|
| 100 | feature | 3700 | 595.66 | 623.00 | 638.67 | 2895.27 | 174 |
| 100 | matching | 3700 | 5.34 | 12.24 | 12.67 | 208.44 | 4 |
| 100 | iteration | 3700 | 605.24 | 631.83 | 646.04 | 3334.38 | n/a |
| 100 | post-processing (per walk) | 37 | 458.1 | 701.3 | 720.2 | 722.3 | 960 |
| 50 | feature | 3900 | 595.11 | 626.71 | 639.69 | 1934.90 | 174 |
| 50 | matching | 3900 | 5.35 | 12.32 | 13.00 | 106.16 | 4 |
| 50 | iteration | 3900 | 604.71 | 636.14 | 646.98 | 2145.10 | n/a |
| 50 | post-processing (per walk) | 78 | 466.7 | 582.5 | 681.9 | 693.0 | 960 |
| 35 | feature | 3885 | 596.41 | 612.07 | 642.91 | 1021.72 | 174 |
| 35 | matching | 3885 | 5.24 | 12.25 | 13.51 | 111.58 | 4 |
| 35 | iteration | 3885 | 605.87 | 621.62 | 650.02 | 1240.14 | n/a |
| 35 | post-processing (per walk) | 111 | 454.7 | 540.5 | 661.1 | 675.9 | 960 |
| 20 | feature | 4000 | 595.99 | 611.63 | 642.19 | 906.77 | 174 |
| 20 | matching | 4000 | 5.30 | 12.26 | 13.10 | 129.35 | 4 |
| 20 | iteration | 4000 | 605.49 | 619.20 | 649.93 | 1151.11 | n/a |
| 20 | post-processing (per walk) | 200 | 448.4 | 529.4 | 625.7 | 645.4 | 960 |
| 15 | feature | 4020 | 596.42 | 612.19 | 641.66 | 907.64 | 174 |
| 15 | matching | 4020 | 5.32 | 12.35 | 13.68 | 127.31 | 4 |
| 15 | iteration | 4020 | 605.72 | 620.59 | 650.73 | 1147.83 | n/a |
| 15 | post-processing (per walk) | 268 | 455.0 | 541.0 | 635.2 | 664.0 | 960 |

## Summary

Main criterion (Table 1 rates and T=100 RMSE): 11 OK, 1 CHECK, 0 STOP.

- T100: CSV matches results.npz (forward 97.3 vs 97.3, post-processed 97.3 vs 97.3)
- T50: CSV matches results.npz (forward 92.3 vs 92.3, post-processed 94.9 vs 94.9)
- T35: CSV matches results.npz (forward 88.3 vs 88.3, post-processed 92.8 vs 92.8)
- T20: CSV matches results.npz (forward 70.5 vs 70.5, post-processed 86.5 vs 86.5)
- T15: CSV matches results.npz (forward 65.3 vs 65.3, post-processed 81.3 vs 81.3)


Saved: /content/drive/MyDrive/Thesis/E2/results/unloc_v1/E2_table.md
